# Week 2 — Data dictionary, coding, and provenance

### Learning goals
- Build a **data dictionary** (column meaning, type, codes)
- Recode categorical variables into readable labels
- Identify what 'missing' could mean and how it affects conclusions

### Deliverable
- Completed data dictionary table
- Short provenance paragraph + 2 caveats


In [ ]:
# Core imports (kept minimal for beginners)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 120)

# Dataset URL (UCI Heart Disease - Cleveland)
UCI_URL = "https://www.archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"

# Column names for processed.cleveland.data (14 columns commonly used in teaching)
COLS = ["age","sex","cp","trestbps","chol","fbs","restecg","thalach",
        "exang","oldpeak","slope","ca","thal","num"]


In [ ]:
import ssl
import io
import urllib.request # Added this import

def load_raw():
    # Create an unverified SSL context to bypass certificate verification
    ctx = ssl.create_default_context()
    ctx.check_hostname = False
    ctx.verify_mode = ssl.CERT_NONE

    # Open the URL with the unverified context
    with urllib.request.urlopen(UCI_URL, context=ctx) as url_response:
        # Read the content and decode it
        s = url_response.read().decode('utf-8')

    # Use io.StringIO to make the string behave like a file for pandas.read_csv
    df_raw = pd.read_csv(io.StringIO(s), header=None, names=COLS)
    return df_raw

def coerce_types(df_raw):
    # Missing values are sometimes encoded as "?"
    df = df_raw.replace("?", np.nan).copy()

    # Convert numeric-looking columns
    numeric_cols = ["age","trestbps","chol","thalach","oldpeak","ca","thal","num"]
    for c in numeric_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    # Binary target: disease present if num > 0 (UCI convention)
    df["disease"] = (df["num"] > 0).astype("Int64")
    return df

df_raw = load_raw()
df = coerce_types(df_raw)

df.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,num,disease
0,63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0,0,0
1,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0,2,1
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0,1,1
3,37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,0.0,3.0,0,0
4,41.0,0.0,2.0,130.0,204.0,0.0,2.0,172.0,0.0,1.4,1.0,0.0,3.0,0,0


In [ ]:
def quick_profile(df):
    """Small, beginner-friendly dataframe profile."""
    display(df.head())
    print("\nShape:", df.shape)
    print("\nDtypes:")
    print(df.dtypes)
    print("\nMissingness (top 10):")
    miss = df.isna().mean().sort_values(ascending=False)
    display(miss.head(10).to_frame("missing_fraction"))
    print("\nBasic describe (numeric):")
    display(df.describe(include=[np.number]).T)

def check_columns(df):
    assert df.columns.is_unique, "Duplicate column names found"
    assert df.shape[0] > 0, "No rows loaded"
    return True


In [ ]:
quick_profile(df)

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,num,disease
0,63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0,0,0
1,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0,2,1
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0,1,1
3,37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,0.0,3.0,0,0
4,41.0,0.0,2.0,130.0,204.0,0.0,2.0,172.0,0.0,1.4,1.0,0.0,3.0,0,0



Shape: (303, 15)

Dtypes:
age         float64
sex         float64
cp          float64
trestbps    float64
chol        float64
fbs         float64
restecg     float64
thalach     float64
exang       float64
oldpeak     float64
slope       float64
ca          float64
thal        float64
num           int64
disease       Int64
dtype: object

Missingness (top 10):


,missing_fraction
ca,0.013201
thal,0.006601
age,0.000000
sex,0.000000
cp,0.000000
trestbps,0.000000
chol,0.000000
fbs,0.000000
restecg,0.000000
thalach,0.000000



Basic describe (numeric):


,count,mean,std,min,25%,50%,75%,max
age,303.0,54.438944,9.038662,29.0,48.0,56.0,61.0,77.0
sex,303.0,0.679868,0.467299,0.0,0.0,1.0,1.0,1.0
cp,303.0,3.158416,0.960126,1.0,3.0,3.0,4.0,4.0
trestbps,303.0,131.689769,17.599748,94.0,120.0,130.0,140.0,200.0
chol,303.0,246.693069,51.776918,126.0,211.0,241.0,275.0,564.0
fbs,303.0,0.148515,0.356198,0.0,0.0,0.0,0.0,1.0
restecg,303.0,0.990099,0.994971,0.0,0.0,1.0,2.0,2.0
thalach,303.0,149.607261,22.875003,71.0,133.5,153.0,166.0,202.0
exang,303.0,0.326733,0.469794,0.0,0.0,0.0,1.0,1.0
oldpeak,303.0,1.039604,1.161075,0.0,0.0,0.8,1.6,6.2


## Step 1 — Data dictionary template (fill this table)
Complete: `meaning`, `type`, `codes_or_units`, `notes`.


In [3]:
import pandas as pd

data_dict = pd.DataFrame({
    "column": ["age", "sex", "cp", "trestbps", "chol", "fbs", "restecg", "thalach", "exang", "oldpeak", "slope", "ca", "thal", "num", "disease"],
    "meaning": [
        "Age (years)",
        "Sex",
        "Chest pain type",
        "Resting blood pressure (mm Hg)",
        "Serum cholesterol (mg/dl)",
        "Fasting blood sugar > 120 mg/dl",
        "Resting ECG results",
        "Maximum heart rate achieved",
        "Exercise-induced angina",
        "ST depression induced by exercise",
        "Slope of the peak exercise ST segment",
        "Number of major vessels colored by fluoroscopy",
        "Thalassemia (blood flow defect)",
        "Original disease severity",
        "Binary disease (num > 0)"
    ],
    "type": [
        "numeric", "categorical", "categorical", "numeric", "numeric", "categorical", "categorical", "numeric", "categorical", "numeric", "categorical", "numeric", "categorical", "numeric", "binary"
    ],
    "codes_or_units": [
        "years", "0=female,1=male", "1=typical angina,2=atypical angina,3=non-anginal pain,4=asymptomatic", "mm Hg", "mg/dl", "0=false,1=true", "0=normal,1=ST-T abnormality,2=hypertrophy", "bpm", "0=no,1=yes", "mm", "1=up-sloping,2=flat,3=down-sloping", "0-3 (count)", "3=normal,6=fixed defect,7=reversible defect", "0-4 (0=no disease)", "0=healthy,1=diseased"
    ],
    "notes": [
        "Complete data", "Male-dominated (68%)", "Most common: asymptomatic (code 4)", "Possible measurement noise", "Outlier: 564 mg/dl", "Mostly false (85%)", "Most common: normal (code 0)", "From exercise test", "Strong predictor", "Continuous variable", "Flat slope most common", "Missing values (1.3%)", "Missing values (0.66%), 7=reversible", "Original outcome", "Derived variable"
    ]
})
data_dict

,column,meaning,type,codes_or_units,notes
0,age,Age (years),numeric,years,Complete data
1,sex,Sex,categorical,"0=female,1=male",Male-dominated (68%)
2,cp,Chest pain type,categorical,"1=typical angina,2=atypical angina,3=non-angin...",Most common: asymptomatic (code 4)
3,trestbps,Resting blood pressure (mm Hg),numeric,mm Hg,Possible measurement noise
4,chol,Serum cholesterol (mg/dl),numeric,mg/dl,Outlier: 564 mg/dl
5,fbs,Fasting blood sugar > 120 mg/dl,categorical,"0=false,1=true",Mostly false (85%)
6,restecg,Resting ECG results,categorical,"0=normal,1=ST-T abnormality,2=hypertrophy",Most common: normal (code 0)
7,thalach,Maximum heart rate achieved,numeric,bpm,From exercise test
8,exang,Exercise-induced angina,categorical,"0=no,1=yes",Strong predictor
9,oldpeak,ST depression induced by exercise,numeric,mm,Continuous variable


## Step 2 — Inspect coded categoricals


In [ ]:
coded_cols = ["sex","cp","fbs","restecg","exang","slope","ca","thal","disease"]
for c in coded_cols:
    print("\n", c)
    display(df[c].value_counts(dropna=False).sort_index())



 sex


sex
0.0     97
1.0    206
Name: count, dtype: int64


 cp


cp
1.0     23
2.0     50
3.0     86
4.0    144
Name: count, dtype: int64


 fbs


...

ca
0.0    176
1.0     65
2.0     38
3.0     20
NaN      4
Name: count, dtype: int64


 thal


thal
3.0    166
6.0     18
7.0    117
NaN      2
Name: count, dtype: int64


 disease


disease
0    164
1    139
Name: count, dtype: Int64

## Step 3 — Create human-readable labels (starter mappings)
Treat these as assumptions unless you verify from documentation.


In [ ]:
sex_map = {0: "female", 1: "male"}
exang_map = {0: "no", 1: "yes"}
fbs_map = {0: "false", 1: "true"}  # fasting blood sugar > 120 mg/dl
cp_map = {1: "typical angina", 2: "atypical angina", 3: "non-anginal pain", 4: "asymptomatic"}

df_labeled = df.copy()
df_labeled["sex_label"] = df_labeled["sex"].map(sex_map)
df_labeled["exang_label"] = df_labeled["exang"].map(exang_map)
df_labeled["fbs_label"] = df_labeled["fbs"].map(fbs_map)
df_labeled["cp_label"] = df_labeled["cp"].map(cp_map)

df_labeled[["sex","sex_label","cp","cp_label","exang","exang_label","fbs","fbs_label"]].head(10)


,sex,sex_label,cp,cp_label,exang,exang_label,fbs,fbs_label
0,1.0,male,1.0,typical angina,0.0,no,1.0,true
1,1.0,male,4.0,asymptomatic,1.0,yes,0.0,false
2,1.0,male,4.0,asymptomatic,1.0,yes,0.0,false
3,1.0,male,3.0,non-anginal pain,0.0,no,0.0,false
4,0.0,female,2.0,atypical angina,0.0,no,0.0,false
5,1.0,male,2.0,atypical angina,0.0,no,0.0,false
6,0.0,female,4.0,asymptomatic,0.0,no,0.0,false
7,0.0,female,4.0,asymptomatic,1.0,yes,0.0,false
8,1.0,male,4.0,asymptomatic,0.0,no,0.0,false
9,1.0,male,4.0,asymptomatic,1.0,yes,1.0,true


# TODO — Provenance & caveats

- *Source:*  
  UCI Machine Learning Repository – Heart Disease Dataset (Cleveland). Original studies: Hungarian Institute of Cardiology, Budapest; University Hospital, Zurich; University Hospital, Basel; V.A. Medical Center, Long Beach and Cleveland Clinic Foundation (1988-1989).

- *Likely population:*  
  Patients presenting with chest pain at the Cleveland Clinic, predominantly male (68%) and middle‑aged to elderly (mean age 54.4 years). This is a clinical population referred for heart disease diagnosis – not the general population.

- *Caveat 1 (Selection bias):*  
  Data comes from a single tertiary hospital, so more severe cases and already diagnosed individuals may be overrepresented. Results may not generalise to primary care or asymptomatic populations.

- *Caveat 2 (Missing data & coding):*  
  Missing values in ca (1.3%) and thal (0.66%) were originally coded as "?" and converted to np.nan. Whether these are truly missing at random is unknown. Also, thal uses codes 3, 6, 7 (6 = fixed defect, 7 = reversible defect) – this coding should be verified from original documentation before use.
